In [0]:
data_df=spark.sql("""select latitude, longitude, MARKET_NAME from pricing_analytics.silver.geolocation_date limit 100""")

In [0]:
HTTPWebSourceRootURL='https://archive-api.open-meteo.com/v1/archive?'
HTTPWebSourceEndURL='&start_date=2023-01-01&end_date=2024-01-01&daily=temperature_2m_max,temperature_2m_min,rain_sum'

In [0]:
import requests
api_endpoints=[]
for names in data_df.collect():
    HTTPWebSourceURL=f"{HTTPWebSourceRootURL}latitude={names['latitude']}&longitude={names['longitude']}{HTTPWebSourceEndURL}"

    api_endPoint=requests.get(HTTPWebSourceURL).json()

    if isinstance(api_endPoint, dict):
        api_endPoint['MARKET_NAME']=names['MARKET_NAME']

        api_endpoints.append(api_endPoint)

In [0]:
import pandas as pds
from pyspark.sql.functions import current_timestamp

pandas_df=pds.DataFrame(api_endpoints)

spark_df=spark.createDataFrame(pandas_df)

In [0]:
final_df=spark_df.withColumn("file_ingestion_date", current_timestamp()).withColumn('file_updated_date', current_timestamp())

In [0]:
final_df.write.format('delta').option('header', True).mode('overwrite').saveAsTable("pricing_analytics.bronze.weather_data")